# Create, update, ACL, and delete a deal

`/admin/deals` stands in for a salesperson. The connector reads the cursor
feed about once a minute and writes `raw/sales/deal/{id}.json`. A delete is an
event on that feed. `commission_rate` stays in the harness and is not in the
snapshot.

Requires `make up`. Cleanup deletes the demo deal.


In [ ]:
import sys
from pathlib import Path

here = Path.cwd()
if not (here / "source_demo.py").exists():
    here = here / "notebooks"
if str(here) not in sys.path:
    sys.path.insert(0, str(here))

import source_demo
sales = source_demo.sales


In [ ]:
DEAL_ID = "deal-demo-notebook"
KEY = sales.key(DEAL_ID, "json")
PAYLOAD = {
    "deal_id": DEAL_ID,
    "account_id": "acc-demo-notebook",
    "account_name": "Notebook Account",
    "arr": 220000,
    "quarter": "2026-Q3",
    "stage": "negotiation",
    "owner_email": "ada@sales.example",
    "product_key": "checkout",
    "native_acl": ["account_team"],
}
sales.delete_if_present("deals", DEAL_ID)
created = sales.admin("POST", "/admin/deals", PAYLOAD)
print(created["deal_id"], "version", created["version"])
sales.expect_status("POST", "/admin/deals", 409, PAYLOAD)
sales.expect_status("POST", "/admin/deals", 422, {**PAYLOAD, "deal_id": "nope", "stage": "nope"})


In [ ]:
def written():
    head = source_demo.snapshot(KEY)
    if head is None:
        return False
    body = source_demo.object_body(KEY)
    return b"Notebook Account" in body and b"commission_rate" not in body

source_demo.wait_until(written, source_demo.WRITE_WAIT, "deal snapshot")
print("snapshot", KEY, "versions", source_demo.versions(KEY))


In [ ]:
before = source_demo.versions(KEY)
edited = sales.admin("PATCH", f"/admin/deals/{DEAL_ID}", {"arr": 230000})
print("version", edited["version"])

def changed():
    return source_demo.versions(KEY) > before and b"230000" in source_demo.object_body(KEY)

source_demo.wait_until(changed, source_demo.WRITE_WAIT, "deal edit")


In [ ]:
changed = sales.admin("PUT", f"/admin/deals/{DEAL_ID}/acl", {"native_acl": ["sales_leadership"]})
print("acl", changed["native_acl"], "version", changed["version"])

def acl_stored():
    head = source_demo.snapshot(KEY)
    return head is not None and "sales_leadership" in head["Metadata"].get(source_demo.META_ACL, "")

source_demo.wait_until(acl_stored, source_demo.WRITE_WAIT, "deal acl")


In [ ]:
sales.admin("DELETE", f"/admin/deals/{DEAL_ID}")
source_demo.wait_until(
    lambda: source_demo.snapshot(KEY) is None, source_demo.GONE_WAIT, "deal deleted"
)
print("deleted", DEAL_ID)
